# 04. Gunduz gallery и retrieval benchmark

Из detector test images формируются непересекающиеся support gallery и query. Оцениваются known, unseen species и unseen genus; название нового таксона всегда приходит из подписанного support-примера.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from core.notebook_runtime import bootstrap_notebook, describe_runtime, gpu_preflight, run_guarded

GPU_INDEX = 0
WORKERS = 4
EVAL_BATCH_SIZE = 32
context = bootstrap_notebook(gpu_index=GPU_INDEX)
PROJECT_ROOT = context.project_root
CONFIG = PROJECT_ROOT / 'configs/classifier_benchmark.yaml'
CHECKPOINT = PROJECT_ROOT / 'artifacts/classifier/dinov2/best.pt'
GALLERY_DIR = PROJECT_ROOT / 'artifacts/classifier/gunduz-gallery'
RUN_GALLERY_BUILD = False
RUN_RETRIEVAL_TEST = False
ENABLE_CLEARML = False
describe_runtime(context)
gpu_preflight(context, minimum_vram_gb=8.0)

## Проверка протокола

In [ ]:
from core.config_loader import load_config
overrides = [f'loader.num_workers={WORKERS}', f'loader.eval_batch_size={EVAL_BATCH_SIZE}']
config = load_config(CONFIG, overrides=overrides)
table_path = PROJECT_ROOT / config['dataset']['table_path']
assert table_path.is_file(), table_path
benchmark = pd.read_csv(table_path)
display(benchmark.groupby('split').agg(crops=('id_crop', 'size'), genera=('genus', 'nunique'), species=('species', 'nunique')).reset_index())
if 'protocol_target' in benchmark.columns:
    display(pd.crosstab(benchmark['protocol_target'], benchmark['split']))
unit_column = 'image_id'
assert benchmark.groupby(unit_column)['split'].nunique().max() == 1, 'Один Gunduz image попал в gallery и query'
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

## Построение FAISS gallery

In [ ]:
gallery_command = context.module_command('scripts.run_build_gallery', '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT), '--output', str(GALLERY_DIR))
for value in overrides:
    gallery_command += ['--set', value]
if RUN_GALLERY_BUILD:
    assert CHECKPOINT.is_file(), CHECKPOINT
run_guarded(context, gallery_command, enabled=RUN_GALLERY_BUILD, label='gallery-build')

## Retrieval test

In [ ]:
test_command = context.module_command('scripts.run_test_classifier', '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT))
for value in overrides:
    test_command += ['--set', value]
if ENABLE_CLEARML:
    test_command += ['--set', 'clearml.enabled=true']
if RUN_RETRIEVAL_TEST:
    assert CHECKPOINT.is_file(), CHECKPOINT
    assert (GALLERY_DIR / 'genus_index.npz').is_file()
    assert (GALLERY_DIR / 'species_index.npz').is_file()
run_guarded(context, test_command, enabled=RUN_RETRIEVAL_TEST, label='retrieval-test')

## Отчёты

In [ ]:
output = PROJECT_ROOT / config['paths']['output_dir']
for path in sorted(output.rglob('*')) if output.exists() else []:
    if path.is_file(): print(path.relative_to(PROJECT_ROOT))